In [1]:
!pip install hampel plotly nbformat

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

# 1. Load from raw CSV

In [2]:
from pathlib import Path
import pandas as pd

# # Tìm file CSV CSI mới nhất trong thư mục hiện tại
# csv_files = sorted(Path('.').glob('csi_data_*.csv'), key=lambda p: p.stat().st_mtime)

# if not csv_files:
#     raise FileNotFoundError("Không tìm thấy file dạng csi_data_*.csv trong thư mục hiện tại")

# latest_csv = csv_files[-1]

latest_csv = 'Router/csi_data_20260305_101328.csv'
print(f"Đang đọc: {latest_csv}")

df = pd.read_csv(latest_csv)

print("\nKích thước dữ liệu:", df.shape)
print("\nTên cột:")
print(df.columns.tolist())

print("\n5 dòng đầu:")
display(df.head())

Đang đọc: Router/csi_data_20260305_101328.csv

Kích thước dữ liệu: (4000, 15)

Tên cột:
['type', 'id', 'mac', 'rssi', 'rate', 'noise_floor', 'fft_gain', 'agc_gain', 'channel', 'local_timestamp', 'sig_len', 'rx_state', 'len', 'first_word', 'data']

5 dòng đầu:


,type,id,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
0,CSI_DATA,89982,98:4a:6b:31:4a:10,-60,11,-99,36,37,11,912442272,83,0,256,0,"[0,0,0,0,0,0,0,0,-73,-25,-67,-32,-63,-46,-63,-..."
1,CSI_DATA,89983,98:4a:6b:31:4a:10,-60,11,-99,-38,55,11,912458702,83,0,256,0,"[0,0,0,0,0,0,0,0,-30,24,-34,20,-39,14,-41,10,-..."
2,CSI_DATA,89984,98:4a:6b:31:4a:10,-53,11,-99,11,37,11,912467861,83,0,256,0,"[0,0,0,0,0,0,0,0,-39,-71,-31,-71,-27,-79,-19,-..."
3,CSI_DATA,89985,98:4a:6b:31:4a:10,-60,11,-99,-40,56,11,912478128,83,0,256,0,"[0,0,0,0,0,0,0,0,-31,-21,-29,-25,-25,-29,-23,-..."
4,CSI_DATA,89986,98:4a:6b:31:4a:10,-53,11,-99,15,37,11,912487946,83,0,256,0,"[0,0,0,0,0,0,0,0,56,-60,63,-49,67,-39,74,-39,7..."


# 2. Calculate Amplitude & phase

In [3]:
import json
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import os
import time

# Parse chuỗi JSON trong cột `data` -> mảng số
def parse_csi_data(value):
    if pd.isna(value):
        return np.array([], dtype=np.float64)
    if isinstance(value, list):
        return np.asarray(value, dtype=np.float64)
    try:
        return np.asarray(json.loads(value), dtype=np.float64)
    except Exception:
        return np.array([], dtype=np.float64)

# Dữ liệu CSI format C5/C6: [imag0, real0, imag1, real1, ...]
def compute_amplitude_phase(csi_raw):
    if csi_raw.size < 2:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)

    imag = csi_raw[0::2]
    real = csi_raw[1::2]

    length = min(len(real), len(imag))
    real = real[:length]
    imag = imag[:length]

    amplitude = np.sqrt(real**2 + imag**2)
    phase = np.arctan2(imag, real)
    return amplitude, phase

def process_packet(data_raw):
    """Parse CSI data và tính amplitude/phase cho 1 packet."""
    csi_array = parse_csi_data(data_raw)
    amp, phs = compute_amplitude_phase(csi_array)
    return csi_array, amp, phs

# Tính cho toàn bộ dataframe với parallel processing
df_calc = df.copy()

print(f"Tính Amplitude & Phase với thread-based parallel processing...")
print(f"Tổng {len(df_calc)} packets\n")

num_workers = max(1, os.cpu_count() - 1)
print(f"Dùng {num_workers} threads...\n")

start_time = time.time()

# Parallel processing với ThreadPoolExecutor (tránh pickle issues)
csi_raw_list = []
amplitude_list = []
phase_list = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(process_packet, raw) for raw in df_calc["data"]]
    
    completed = 0
    for future in futures:
        try:
            csi_array, amp, phs = future.result()
            csi_raw_list.append(csi_array)
            amplitude_list.append(amp)
            phase_list.append(phs)
            completed += 1
            if completed % max(1, len(df_calc)//10) == 0:
                print(f"  Đã xử lý {completed}/{len(df_calc)} packets...")
        except Exception as e:
            print(f"  Lỗi xử lý packet: {e}")
            csi_raw_list.append(np.array([]))
            amplitude_list.append(np.array([]))
            phase_list.append(np.array([]))
            completed += 1

elapsed = time.time() - start_time

df_calc["csi_raw"] = csi_raw_list
df_calc["amplitude"] = amplitude_list
df_calc["phase"] = phase_list

print(f"Hoàn tất tính amplitude & phase (took {elapsed:.2f}s)\n")

# Feature thống kê nhanh theo từng packet
df_calc["subcarrier_count"] = df_calc["amplitude"].apply(len)
df_calc["amp_mean"] = df_calc["amplitude"].apply(lambda x: float(np.mean(x)) if len(x) else np.nan)
df_calc["amp_std"] = df_calc["amplitude"].apply(lambda x: float(np.std(x)) if len(x) else np.nan)
df_calc["phase_mean"] = df_calc["phase"].apply(lambda x: float(np.mean(x)) if len(x) else np.nan)
df_calc["phase_std"] = df_calc["phase"].apply(lambda x: float(np.std(x)) if len(x) else np.nan)

print("Hoàn tất tính feature thống kê")
print("Số packet:", len(df_calc))
print("Subcarrier/packet (min/max):", int(df_calc["subcarrier_count"].min()), "/", int(df_calc["subcarrier_count"].max()))

display(
    df_calc[["rssi", "len", "subcarrier_count", "amp_mean", "amp_std", "phase_mean", "phase_std"]].head()
 )

# Xem chi tiết packet đầu tiên
first_idx = df_calc["subcarrier_count"].gt(0).idxmax()
print(f"\nPacket mẫu index: {first_idx}")
print("5 amplitude đầu:", df_calc.loc[first_idx, "amplitude"][:5])
print("5 phase đầu:", df_calc.loc[first_idx, "phase"][:5])

Tính Amplitude & Phase với thread-based parallel processing...
Tổng 4000 packets

Dùng 71 threads...

  Đã xử lý 400/4000 packets...
  Đã xử lý 800/4000 packets...
  Đã xử lý 1200/4000 packets...
  Đã xử lý 1600/4000 packets...
  Đã xử lý 2000/4000 packets...
  Đã xử lý 2400/4000 packets...
  Đã xử lý 2800/4000 packets...
  Đã xử lý 3200/4000 packets...
  Đã xử lý 3600/4000 packets...
  Đã xử lý 4000/4000 packets...
Hoàn tất tính amplitude & phase (took 0.65s)

Hoàn tất tính feature thống kê
Số packet: 4000
Subcarrier/packet (min/max): 128 / 128


,rssi,len,subcarrier_count,amp_mean,amp_std,phase_mean,phase_std
0,-60,256,128,51.624630,31.201573,0.437695,1.854173
1,-60,256,128,25.981632,15.717777,0.156273,1.759956
2,-53,256,128,53.161815,33.478205,0.502001,1.631484
3,-60,256,128,25.039728,15.289023,0.325378,1.650024
4,-53,256,128,53.401310,32.897024,0.216382,1.371926



Packet mẫu index: 0
5 amplitude đầu: [ 0.         0.         0.         0.        77.1621669]
5 phase đầu: [ 0.          0.          0.          0.         -1.90074342]


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Chọn packet đầu tiên có dữ liệu hợp lệ
valid_packets = df_calc[df_calc["subcarrier_count"] > 0]
if valid_packets.empty:
    raise ValueError("Không có packet hợp lệ để plot")

sample_idx = valid_packets.index[0]
amp = np.asarray(df_calc.loc[sample_idx, "amplitude"])
phs = np.asarray(df_calc.loc[sample_idx, "phase"])
subcarrier_idx = np.arange(len(amp))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f"Amplitude - Packet {sample_idx}",
                                    f"Phase - Packet {sample_idx}"])

fig.add_trace(go.Scatter(x=subcarrier_idx, y=amp, mode='lines',
                          line=dict(color='royalblue', width=1.2),
                          name='Amplitude'), row=1, col=1)

fig.add_trace(go.Scatter(x=subcarrier_idx, y=phs, mode='lines',
                          line=dict(color='darkorange', width=1.2),
                          name='Phase'), row=2, col=1)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Subcarrier Index", row=2, col=1)
fig.update_layout(height=700, width=1200, showlegend=True,
                  template='plotly_white')
fig.show()

# 3. Extract Subcarrier 10 Time Series

In [5]:
import numpy as np
import pandas as pd

print(f"Trích xuất tất cả subcarrier từ tất cả packet...\n")

# Tạo dictionary chứa time series của từng subcarrier
ts_all = {}

# Xác định số subcarrier tối đa
max_subcarrier = int(df_calc["subcarrier_count"].max())

# Loại bỏ guard subcarriers: 4 đầu + 3 cuối
GUARD_FRONT = 4
GUARD_BACK = 3
data_start = GUARD_FRONT
data_end = max_subcarrier - GUARD_BACK

print(f"Số subcarrier gốc: {max_subcarrier}")
print(f"Bỏ {GUARD_FRONT} guard đầu (0..{GUARD_FRONT-1}) + {GUARD_BACK} guard cuối ({data_end}..{max_subcarrier-1})")
print(f"Trích xuất subcarrier {data_start}..{data_end-1} ({data_end - data_start} data subcarriers)\n")

# Trích xuất từng subcarrier (chỉ data, bỏ guard)
for subcarrier_idx in range(data_start, data_end):
    timeseries_data = []
    
    for packet_idx in df_calc.index:
        amp = np.asarray(df_calc.loc[packet_idx, "amplitude"])
        phs = np.asarray(df_calc.loc[packet_idx, "phase"])
        
        if len(amp) <= subcarrier_idx:
            continue
        
        timeseries_data.append({
            "packet_idx": packet_idx,
            "amp": amp[subcarrier_idx],
            "phase": phs[subcarrier_idx],
        })
    
    if len(timeseries_data) > 0:
        ts_all[subcarrier_idx] = pd.DataFrame(timeseries_data)

print(f"Trích xuất thành công {len(ts_all)} subcarrier (đã bỏ guard)")
print(f"Số packet per subcarrier: min={min(len(ts_all[k]) for k in ts_all)}, max={max(len(ts_all[k]) for k in ts_all)}")
print(f"Subcarrier range: [{min(ts_all.keys())} ... {max(ts_all.keys())}]")

# Set SUBCARRIER_INDEX cho phần visualization sau này
SUBCARRIER_INDEX = 10
if SUBCARRIER_INDEX not in ts_all:
    SUBCARRIER_INDEX = list(ts_all.keys())[0]
    print(f"Subcarrier 10 không có đủ dữ liệu, sử dụng subcarrier {SUBCARRIER_INDEX}")

print(f"\nThống kê gốc Subcarrier {SUBCARRIER_INDEX}:")
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"  Amplitude: mean={ts_sample['amp'].mean():.3f}, std={ts_sample['amp'].std():.3f}")
print(f"  Phase: mean={ts_sample['phase'].mean():.3f}, std={ts_sample['phase'].std():.3f}")

Trích xuất tất cả subcarrier từ tất cả packet...

Số subcarrier gốc: 128
Bỏ 4 guard đầu (0..3) + 3 guard cuối (125..127)
Trích xuất subcarrier 4..124 (121 data subcarriers)

Trích xuất thành công 121 subcarrier (đã bỏ guard)
Số packet per subcarrier: min=4000, max=4000
Subcarrier range: [4 ... 124]

Thống kê gốc Subcarrier 10:
  Amplitude: mean=49.187, std=17.126
  Phase: mean=0.023, std=1.825


In [6]:
import numpy as np
import pandas as pd
from hampel import hampel
from concurrent.futures import ProcessPoolExecutor
import os
import time

# Tham số Hampel
K_WINDOW = 30 # 200 => 30
N_SIGMA = 3.0

def apply_hampel_single(sub_idx, ts_df, k_window, n_sigma):
    """Hampel filter cho một subcarrier (dùng cho parallel processing)."""
    # Pad data với reflect mode để tránh edge artifacts
    pad_len = k_window * 2
    amp_padded = np.pad(ts_df['amp'].values, (pad_len, pad_len), mode='reflect')
    phs_padded = np.pad(ts_df['phase'].values, (pad_len, pad_len), mode='reflect')
    
    # Hampel filter trên padded data
    amp_hampel_res = hampel(amp_padded, window_size=k_window, n_sigma=n_sigma)
    amp_hampel = amp_hampel_res.filtered_data[pad_len:-pad_len]
    amp_hampel_outliers = np.zeros(len(ts_df), dtype=bool)
    valid_outliers = [idx - pad_len for idx in amp_hampel_res.outlier_indices 
                      if pad_len <= idx < len(amp_padded) - pad_len]
    amp_hampel_outliers[valid_outliers] = True
    
    phs_hampel_res = hampel(phs_padded, window_size=k_window, n_sigma=n_sigma)
    phs_hampel = phs_hampel_res.filtered_data[pad_len:-pad_len]
    phs_hampel_outliers = np.zeros(len(ts_df), dtype=bool)
    valid_outliers = [idx - pad_len for idx in phs_hampel_res.outlier_indices 
                      if pad_len <= idx < len(phs_padded) - pad_len]
    phs_hampel_outliers[valid_outliers] = True
    
    return sub_idx, {
        "amp_hampel": amp_hampel,
        "amp_hampel_outliers": amp_hampel_outliers,
        "phase_hampel": phs_hampel,
        "phase_hampel_outliers": phs_hampel_outliers
    }

print(f"Áp dụng Hampel filter (k={K_WINDOW}, nσ={N_SIGMA}) cho tất cả {len(ts_all)} subcarrier...\n")

# Parallel processing với ThreadPoolExecutor (tránh pickle issues)
num_workers = max(1, os.cpu_count() - 1)
print(f"Dùng {num_workers} threads cho {len(ts_all)} subcarriers...\n")

start_time = time.time()

with ProcessPoolExecutor(max_workers=num_workers)  as executor:
    futures = [
        executor.submit(apply_hampel_single, sub_idx, ts_all[sub_idx], K_WINDOW, N_SIGMA)
        for sub_idx in ts_all.keys()
    ]
    
    completed = 0
    for future in futures:
        try:
            sub_idx, result_dict = future.result()
            for key, val in result_dict.items():
                ts_all[sub_idx][key] = val
            completed += 1
            if completed % max(1, len(ts_all)//10) == 0:
                print(f"  Đã xử lý {completed}/{len(ts_all)} subcarriers...")
        except Exception as e:
            print(f"  Lỗi xử lý subcarrier: {e}")
            completed += 1

elapsed = time.time() - start_time
print(f"Hoàn tất Hampel filter cho tất cả subcarrier (took {elapsed:.2f}s)")

# Hiển thị thống kê cho subcarrier 10
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"\nThống kê sau Hampel - Subcarrier {SUBCARRIER_INDEX}:")
print(f"  Amplitude outliers: {np.sum(ts_sample['amp_hampel_outliers'])} ({100*np.sum(ts_sample['amp_hampel_outliers'])/len(ts_sample):.2f}%)")
print(f"  Phase outliers: {np.sum(ts_sample['phase_hampel_outliers'])} ({100*np.sum(ts_sample['phase_hampel_outliers'])/len(ts_sample):.2f}%)")

Áp dụng Hampel filter (k=30, nσ=3.0) cho tất cả 121 subcarrier...

Dùng 71 threads cho 121 subcarriers...



  Đã xử lý 12/121 subcarriers...
  Đã xử lý 24/121 subcarriers...
  Đã xử lý 36/121 subcarriers...
  Đã xử lý 48/121 subcarriers...
  Đã xử lý 60/121 subcarriers...
  Đã xử lý 72/121 subcarriers...
  Đã xử lý 84/121 subcarriers...
  Đã xử lý 96/121 subcarriers...
  Đã xử lý 108/121 subcarriers...
  Đã xử lý 120/121 subcarriers...
Hoàn tất Hampel filter cho tất cả subcarrier (took 3.47s)

Thống kê sau Hampel - Subcarrier 10:
  Amplitude outliers: 625 (15.62%)
  Phase outliers: 6 (0.15%)


In [7]:
# import numpy as np
# import pandas as pd
# from hampel import hampel

# # Tham số Hampel
# K_WINDOW = 30 # 200 => 30
# N_SIGMA = 3.0

# print(f"Áp dụng Hampel filter (k={K_WINDOW}, nσ={N_SIGMA}) cho tất cả {len(ts_all)} subcarrier...\n")

# # Áp dụng Hampel cho mỗi subcarrier
# for sub_idx in ts_all.keys():
#     ts_df = ts_all[sub_idx]
    
#     # Pad data với reflect mode để tránh edge artifacts
#     pad_len = K_WINDOW * 2
#     amp_padded = np.pad(ts_df['amp'].values, (pad_len, pad_len), mode='reflect')
#     phs_padded = np.pad(ts_df['phase'].values, (pad_len, pad_len), mode='reflect')
    
#     # Hampel filter trên padded data
#     amp_hampel_res = hampel(amp_padded, window_size=K_WINDOW, n_sigma=N_SIGMA)
#     amp_hampel = amp_hampel_res.filtered_data[pad_len:-pad_len]  # Bỏ padding
#     amp_hampel_outliers = np.zeros(len(ts_df), dtype=bool)
#     # Điều chỉnh outlier indices từ padded sang original
#     valid_outliers = [idx - pad_len for idx in amp_hampel_res.outlier_indices 
#                       if pad_len <= idx < len(amp_padded) - pad_len]
#     amp_hampel_outliers[valid_outliers] = True
    
#     phs_hampel_res = hampel(phs_padded, window_size=K_WINDOW, n_sigma=N_SIGMA)
#     phs_hampel = phs_hampel_res.filtered_data[pad_len:-pad_len]  # Bỏ padding
#     phs_hampel_outliers = np.zeros(len(ts_df), dtype=bool)
#     # Điều chỉnh outlier indices từ padded sang original
#     valid_outliers = [idx - pad_len for idx in phs_hampel_res.outlier_indices 
#                       if pad_len <= idx < len(phs_padded) - pad_len]
#     phs_hampel_outliers[valid_outliers] = True
    
#     # Lưu vào dict
#     ts_all[sub_idx]["amp_hampel"] = amp_hampel
#     ts_all[sub_idx]["amp_hampel_outliers"] = amp_hampel_outliers
#     ts_all[sub_idx]["phase_hampel"] = phs_hampel
#     ts_all[sub_idx]["phase_hampel_outliers"] = phs_hampel_outliers

# print(f"Hoàn tất Hampel filter cho tất cả subcarrier")

# # Hiển thị thống kê cho subcarrier 10
# ts_sample = ts_all[SUBCARRIER_INDEX]
# print(f"\nThống kê sau Hampel - Subcarrier {SUBCARRIER_INDEX}:")
# print(f"  Amplitude outliers: {np.sum(ts_sample['amp_hampel_outliers'])} ({100*np.sum(ts_sample['amp_hampel_outliers'])/len(ts_sample):.2f}%)")
# print(f"  Phase outliers: {np.sum(ts_sample['phase_hampel_outliers'])} ({100*np.sum(ts_sample['phase_hampel_outliers'])/len(ts_sample):.2f}%)")

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Chỉ plot subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]
packet_idx = np.arange(len(ts_df))

# Lấy ra index của outlier
amp_outlier_idx = np.where(ts_df['amp_hampel_outliers'])[0]
phs_outlier_idx = np.where(ts_df['phase_hampel_outliers'])[0]

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=[
                        f"Amplitude - Subcarrier {SUBCARRIER_INDEX} (Before Hampel)",
                        f"Amplitude - Subcarrier {SUBCARRIER_INDEX} (After Hampel, k={K_WINDOW}, nσ={N_SIGMA})"
                    ])

# Plot 1: Amplitude - Trước Hampel
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp'], mode='lines',
                          line=dict(color='royalblue', width=1.2),
                          opacity=0.7, name="Original"),
              row=1, col=1)

# Plot 2: Amplitude - Sau Hampel
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_hampel'], mode='lines',
                          line=dict(color='green', width=1.2),
                          opacity=0.7, name="Hampel Filtered"),
              row=2, col=1)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Amplitude", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_layout(height=500, width=1200, showlegend=True,
                  template='plotly_white')
fig.show()

# 4. Savitzky-Golay Filter (Smoothing)

In [9]:
from scipy.signal import savgol_filter
import numpy as np

# Tham số Savitzky-Golay
POLYORDER = 3      # Bậc đa thức
WINDOW_LENGTH = 51 # Kích thước cửa sổ

print(f"Áp dụng Savitzky-Golay Filter (p={POLYORDER}, l={WINDOW_LENGTH}) cho tất cả subcarriers...\n")

# Áp dụng SG filter cho tất cả subcarriers
for sub_idx in ts_all.keys():
    ts_df = ts_all[sub_idx]
    
    amp_len = len(ts_df)
    window_len = min(WINDOW_LENGTH, amp_len if amp_len % 2 == 1 else amp_len - 1)
    
    # Áp dụng SG filter từ amplitude & phase đã qua Hampel
    if window_len >= POLYORDER + 1:
        amp_sg = savgol_filter(ts_df['amp_hampel'].values, window_length=window_len, polyorder=POLYORDER)
    else:
        amp_sg = ts_df['amp_hampel'].values
    
    if window_len >= POLYORDER + 1:
        phs_sg = savgol_filter(ts_df['phase_hampel'].values, window_length=window_len, polyorder=POLYORDER)
    else:
        phs_sg = ts_df['phase_hampel'].values
    
    # Lưu vào dataframe
    ts_all[sub_idx]["amp_sg"] = amp_sg
    ts_all[sub_idx]["phase_sg"] = phs_sg

print(f"Hoàn tất SG filter cho tất cả {len(ts_all)} subcarriers")
print(f"Thông số: polyorder={POLYORDER}, window_length={WINDOW_LENGTH}\n")

# Hiển thị thống kê cho subcarrier tham chiếu
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"Thống kê cho Subcarrier {SUBCARRIER_INDEX} sau Savitzky-Golay:")
print(f"  Amplitude: mean={np.mean(ts_sample['amp_sg']):.3f}, std={np.std(ts_sample['amp_sg']):.3f}")
print(f"  Phase: mean={np.mean(ts_sample['phase_sg']):.3f}, std={np.std(ts_sample['phase_sg']):.3f}")

# So sánh MSE giữa Hampel và SG
amp_mse = np.mean((ts_sample['amp_hampel'].values - ts_sample['amp_sg'].values)**2)
phs_mse = np.mean((ts_sample['phase_hampel'].values - ts_sample['phase_sg'].values)**2)

print(f"\nMSE (Hampel → SG) cho Subcarrier {SUBCARRIER_INDEX}:")
print(f"  Amplitude: {amp_mse:.6f}")
print(f"  Phase: {phs_mse:.6f}")

Áp dụng Savitzky-Golay Filter (p=3, l=51) cho tất cả subcarriers...

Hoàn tất SG filter cho tất cả 121 subcarriers
Thông số: polyorder=3, window_length=51

Thống kê cho Subcarrier 10 sau Savitzky-Golay:
  Amplitude: mean=42.041, std=0.345
  Phase: mean=0.024, std=0.429

MSE (Hampel → SG) cho Subcarrier 10:
  Amplitude: 1.509234
  Phase: 3.153830


# 5. Visualization - Hampel vs Savitzky-Golay vs Final

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Lấy dữ liệu từ dict cho subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]

if ts_df.empty:
    raise ValueError("Time series dataframe trống, không thể vẽ")

packet_idx = np.arange(len(ts_df))

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[
                        f"Amplitude - Original (SC {SUBCARRIER_INDEX})",
                        f"Amplitude - After Hampel (k={K_WINDOW}, nσ={N_SIGMA})",
                        f"Amplitude - After SG (p={POLYORDER}, l={WINDOW_LENGTH})",
                        f"Phase - Original (SC {SUBCARRIER_INDEX})",
                        f"Phase - After Hampel (k={K_WINDOW}, nσ={N_SIGMA})",
                        f"Phase - After SG (p={POLYORDER}, l={WINDOW_LENGTH})",
                    ])

# Row 1: Amplitude
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp'], mode='lines',
                          line=dict(color='royalblue', width=1), opacity=0.7,
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_hampel'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_sg'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=1, col=3)

# Row 2: Phase
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase'], mode='lines',
                          line=dict(color='darkorange', width=1), opacity=0.7,
                          showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_hampel'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=2, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_sg'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=2, col=3)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=2)
fig.update_xaxes(title_text="Packet Index", row=2, col=3)
fig.update_layout(height=600, width=1600, showlegend=False,
                  template='plotly_white')
fig.show()

# 6. Elliptic Bandpass Filter (0.1–0.4 Hz)

In [11]:
import numpy as np
from scipy.signal import ellip, sosfiltfilt

# Giả sử tốc độ thu ~100 packet/s => fs = 100 Hz
FS = 100.0
LOWCUT = 0.1
HIGHCUT = 0.4
ORDER = 2 # 4 => 2
RP = 0.15    # passband ripple (dB)
RS = 40     # stopband attenuation (dB)

nyquist = FS / 2
if HIGHCUT >= nyquist:
    raise ValueError(f"HIGHCUT={HIGHCUT} phải nhỏ hơn Nyquist={nyquist}")

# Thiết kế Elliptic bandpass dưới dạng SOS để ổn định số học
sos = ellip(
    N=ORDER,
    rp=RP,
    rs=RS,
    Wn=[LOWCUT / nyquist, HIGHCUT / nyquist],
    btype='bandpass',
    output='sos'
)

print(f"Áp dụng Elliptic Bandpass Filter ({LOWCUT}-{HIGHCUT} Hz) cho tất cả subcarriers...")

# padlen lớn để sosfiltfilt pad đủ dài, triệt transient tại biên
PADLEN = int(3 / LOWCUT * FS)  # ~3 chu kỳ tần thấp nhất = 3000 samples

print(f"  Edge handling: padlen={PADLEN}\n")

# Áp dụng cho tất cả subcarriers
for sub_idx in ts_all.keys():
    ts_df = ts_all[sub_idx]
    
    # Lấy input từ bước SG nếu có, không thì từ Hampel
    amp_in = ts_df['amp_sg'].values if 'amp_sg' in ts_df.columns else ts_df['amp_hampel'].values
    phs_in = ts_df['phase_sg'].values if 'phase_sg' in ts_df.columns else ts_df['phase_hampel'].values
    
    # padlen lớn giúp giảm transient, cap tại len-1 để tránh lỗi
    pad = min(PADLEN, len(amp_in) - 1)
    
    # Áp dụng filtfilt (zero-phase) với padding tăng cường
    amp_ellip = sosfiltfilt(sos, amp_in, padlen=pad)
    phs_ellip = sosfiltfilt(sos, phs_in, padlen=pad)
    
    # Lưu vào dataframe (không trim, giữ nguyên số mẫu)
    ts_all[sub_idx]['amp_ellip'] = amp_ellip
    ts_all[sub_idx]['phase_ellip'] = phs_ellip

print(f"Hoàn tất Elliptic filter cho tất cả {len(ts_all)} subcarriers")
print(f"Thông số: order={ORDER}, rp={RP}dB, rs={RS}dB, band={LOWCUT}-{HIGHCUT} Hz")
print(f"Số mẫu per subcarrier: {len(ts_all[SUBCARRIER_INDEX])} (không thay đổi)\n")

# Hiển thị thống kê cho tham chiếu subcarrier
ts_sample = ts_all[SUBCARRIER_INDEX]
print(f"Thống kê sau Elliptic Bandpass cho Subcarrier {SUBCARRIER_INDEX}:")
print(f"  Amplitude: mean={np.mean(ts_sample['amp_ellip']):.3f}, std={np.std(ts_sample['amp_ellip']):.3f}")
print(f"  Phase: mean={np.mean(ts_sample['phase_ellip']):.3f}, std={np.std(ts_sample['phase_ellip']):.3f}")

Áp dụng Elliptic Bandpass Filter (0.1-0.4 Hz) cho tất cả subcarriers...
  Edge handling: padlen=3000

Hoàn tất Elliptic filter cho tất cả 121 subcarriers
Thông số: order=2, rp=0.15dB, rs=40dB, band=0.1-0.4 Hz
Số mẫu per subcarrier: 4000 (không thay đổi)

Thống kê sau Elliptic Bandpass cho Subcarrier 10:
  Amplitude: mean=-0.030, std=0.386
  Phase: mean=-0.001, std=0.205


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Hiển thị Elliptic filter effect trên subcarrier 10
ts_df = ts_all[SUBCARRIER_INDEX]
packet_idx = np.arange(len(ts_df))

amp_before = ts_df['amp_sg'].values if 'amp_sg' in ts_df.columns else ts_df['amp_hampel'].values
phs_before = ts_df['phase_sg'].values if 'phase_sg' in ts_df.columns else ts_df['phase_hampel'].values

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[
                        f"Amplitude Before Elliptic (SC {SUBCARRIER_INDEX})",
                        "Amplitude After Elliptic (0.1-0.4 Hz)",
                        f"Phase Before Elliptic (SC {SUBCARRIER_INDEX})",
                        "Phase After Elliptic (0.1-0.4 Hz)",
                    ])

fig.add_trace(go.Scatter(x=packet_idx, y=amp_before, mode='lines',
                          line=dict(color='royalblue', width=1), opacity=0.7,
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['amp_ellip'], mode='lines',
                          line=dict(color='green', width=1), opacity=0.7,
                          showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=packet_idx, y=phs_before, mode='lines',
                          line=dict(color='darkorange', width=1), opacity=0.7,
                          showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=packet_idx, y=ts_df['phase_ellip'], mode='lines',
                          line=dict(color='purple', width=1), opacity=0.7,
                          showlegend=False), row=2, col=2)

fig.update_yaxes(title_text="Amplitude", row=1, col=1)
fig.update_yaxes(title_text="Phase (rad)", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=2)
fig.update_layout(height=600, width=1200, showlegend=False,
                  template='plotly_white')
fig.show()

In [13]:
import plotly.graph_objects as go
import numpy as np

# Plot tất cả subcarriers sau Elliptic filter trên cùng một biểu đồ
num_subcarriers = len(ts_all)

if num_subcarriers == 0:
    raise ValueError("ts_all rỗng, không có subcarrier để vẽ")

fig = go.Figure()

for sub_idx in sorted(ts_all.keys()):
    ts_df = ts_all[sub_idx]
    if 'amp_ellip' not in ts_df.columns:
        continue
    fig.add_trace(go.Scatter(
        x=np.arange(len(ts_df)),
        y=ts_df['amp_ellip'].values,
        mode='lines',
        line=dict(width=0.8),
        opacity=0.65,
        name=f'SC{sub_idx}',
        showlegend=False
    ))

fig.update_layout(
    title=f'All {num_subcarriers} Subcarriers After Elliptic Filter',
    xaxis_title='Packet Index',
    yaxis_title='Amplitude',
    height=600, width=1200,
    template='plotly_white'
)
fig.show()

print(f'Plotted {num_subcarriers} subcarriers after elliptic filter in single plot')

Plotted 121 subcarriers after elliptic filter in single plot


# 7. PCA Dimensionality Reduction - Top 5 Components

In [14]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chuẩn bị ma trận dữ liệu: mỗi hàng là 1 packet, mỗi cột là 1 subcarrier
num_packets = max([len(ts_all[k]) for k in ts_all.keys()])
num_subcarriers = len(ts_all)

# Khởi tạo ma trận amplitude và phase
amp_matrix = np.zeros((num_packets, num_subcarriers))
phs_matrix = np.zeros((num_packets, num_subcarriers))

# Điền dữ liệu từ ts_all (sau khi đã qua filters: Hampel, SG, Elliptic)
for col_idx, sub_idx in enumerate(sorted(ts_all.keys())):
    ts_df = ts_all[sub_idx]
    n_samples = len(ts_df)
    amp_matrix[:n_samples, col_idx] = ts_df['amp_ellip'].values
    phs_matrix[:n_samples, col_idx] = ts_df['phase_ellip'].values

print(f"Ma trận amplitude: {amp_matrix.shape}")
print(f"Ma trận phase: {phs_matrix.shape}")

# Áp dụng PCA cho amplitude (giảm xuống 5 thành phần chính)
N_COMPONENTS = 5
pca_amp = PCA(n_components=N_COMPONENTS)
amp_pca = pca_amp.fit_transform(amp_matrix)

print(f"\nPCA Amplitude:")
print(f"  Input shape: {amp_matrix.shape}")
print(f"  Output shape: {amp_pca.shape}")
print(f"  Explained variance ratio: {pca_amp.explained_variance_ratio_}")
print(f"  Total variance explained: {np.sum(pca_amp.explained_variance_ratio_)*100:.2f}%")

# Áp dụng PCA cho phase
pca_phs = PCA(n_components=N_COMPONENTS)
phs_pca = pca_phs.fit_transform(phs_matrix)

print(f"\nPCA Phase:")
print(f"  Input shape: {phs_matrix.shape}")
print(f"  Output shape: {phs_pca.shape}")
print(f"  Explained variance ratio: {pca_phs.explained_variance_ratio_}")
print(f"  Total variance explained: {np.sum(pca_phs.explained_variance_ratio_)*100:.2f}%")

# Vẽ biểu đồ variance explained
pc_labels = [f'PC{i}' for i in range(1, N_COMPONENTS+1)]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['PCA Amplitude - Variance Explained',
                                    'PCA Phase - Variance Explained'])

# Amplitude variance
fig.add_trace(go.Bar(x=pc_labels, y=pca_amp.explained_variance_ratio_,
                     marker_color='royalblue', opacity=0.7, name='Amp Variance',
                     showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pc_labels, y=np.cumsum(pca_amp.explained_variance_ratio_),
                          mode='lines+markers', line=dict(color='red', width=2),
                          marker=dict(size=8), name='Cumulative'),
              row=1, col=1)

# Phase variance
fig.add_trace(go.Bar(x=pc_labels, y=pca_phs.explained_variance_ratio_,
                     marker_color='darkorange', opacity=0.7, name='Phs Variance',
                     showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=pc_labels, y=np.cumsum(pca_phs.explained_variance_ratio_),
                          mode='lines+markers', line=dict(color='red', width=2),
                          marker=dict(size=8), name='Cumulative',
                          showlegend=False),
              row=1, col=2)

fig.update_yaxes(title_text="Variance Explained Ratio", row=1, col=1)
fig.update_yaxes(title_text="Variance Explained Ratio", row=1, col=2)
fig.update_xaxes(title_text="Principal Component", row=1, col=1)
fig.update_xaxes(title_text="Principal Component", row=1, col=2)
fig.update_layout(height=500, width=1400, template='plotly_white')
fig.show()

print(f"\nPCA hoàn tất! Giảm từ {num_subcarriers} subcarriers xuống {N_COMPONENTS} principal components")

Ma trận amplitude: (4000, 121)
Ma trận phase: (4000, 121)

PCA Amplitude:
  Input shape: (4000, 121)
  Output shape: (4000, 5)
  Explained variance ratio: [0.73171876 0.0819947  0.04324929 0.02603923 0.01888837]
  Total variance explained: 90.19%

PCA Phase:
  Input shape: (4000, 121)
  Output shape: (4000, 5)
  Explained variance ratio: [0.43650305 0.33245197 0.07471422 0.03020115 0.02220276]
  Total variance explained: 89.61%



PCA hoàn tất! Giảm từ 121 subcarriers xuống 5 principal components


# 8. Visualization - Top 5 Principal Components

In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Vẽ 10 thành phần chính
packet_idx = np.arange(len(amp_pca))

colors = ['royalblue', 'darkorange', 'green', 'red', 'purple',
          'brown', 'pink', 'gray', 'olive', 'cyan']

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=[
                        f'Amplitude - Top {N_COMPONENTS} Principal Components (Total: {np.sum(pca_amp.explained_variance_ratio_)*100:.1f}% variance)',
                        f'Phase - Top {N_COMPONENTS} Principal Components (Total: {np.sum(pca_phs.explained_variance_ratio_)*100:.1f}% variance)',
                    ])

# Plot Amplitude - skip PC1 (i starts from 1)
for i in range(1, N_COMPONENTS):
    fig.add_trace(go.Scatter(
        x=packet_idx, y=amp_pca[:, i], mode='lines',
        line=dict(color=colors[i], width=1.5), opacity=0.8,
        name=f'PC{i+1} ({pca_amp.explained_variance_ratio_[i]*100:.1f}%)',
        legendgroup='amp', legendgrouptitle_text='Amplitude'
    ), row=1, col=1)

# Plot Phase - all PCs
for i in range(N_COMPONENTS):
    fig.add_trace(go.Scatter(
        x=packet_idx, y=phs_pca[:, i], mode='lines',
        line=dict(color=colors[i], width=1.5), opacity=0.8,
        name=f'PC{i+1} ({pca_phs.explained_variance_ratio_[i]*100:.1f}%)',
        legendgroup='phs', legendgrouptitle_text='Phase'
    ), row=2, col=1)

fig.update_yaxes(title_text="Amplitude (PCA)", row=1, col=1)
fig.update_yaxes(title_text="Phase (PCA)", row=2, col=1)
fig.update_xaxes(title_text="Packet Index", row=1, col=1)
fig.update_xaxes(title_text="Packet Index", row=2, col=1)
fig.update_layout(height=600, width=1200, template='plotly_white')
fig.show()

print(f"\nPrincipal Components Ready:")
print(f"  amp_pca: shape {amp_pca.shape} - {N_COMPONENTS} principal components from amplitude")
print(f"  phs_pca: shape {phs_pca.shape} - {N_COMPONENTS} principal components from phase")


Principal Components Ready:
  amp_pca: shape (4000, 5) - 5 principal components from amplitude
  phs_pca: shape (4000, 5) - 5 principal components from phase


# 9. Power Spectrogram (STFT) Feature Extraction for CNN

In [16]:
import numpy as np
from scipy.signal import spectrogram

# ============ Tham số Spectrogram ============
FS = 100.0          # Sampling frequency (Hz)
NPERSEG = 500       # Window length = 5s × 100 Hz
NOVERLAP = 450      # Overlap 90%
WINDOW = 'hann'     # Cửa sổ Hann

# Dải tần nhịp thở
BREATH_FREQ_MIN = 0.1   # Hz
BREATH_FREQ_MAX = 0.6   # Hz

print(f"Tính Power Spectrogram (STFT) cho {N_COMPONENTS} principal components...")
print(f"  Window: {WINDOW}, nperseg={NPERSEG} ({NPERSEG/FS:.1f}s)")
print(f"  Overlap: {NOVERLAP} ({NOVERLAP/NPERSEG*100:.0f}%)")
print(f"  Hop size: {NPERSEG - NOVERLAP} samples ({(NPERSEG - NOVERLAP)/FS:.2f}s)")
print(f"  Breathing band: [{BREATH_FREQ_MIN} - {BREATH_FREQ_MAX}] Hz\n")

def compute_spectrogram_matrix(pca_matrix, fs, nperseg, noverlap, window):
    """
    Tính spectrogram cho tất cả PCA components.
    Returns: freqs, times, list of Sxx matrices (one per component)
    """
    spectrograms = []
    freqs = None
    times = None
    
    for comp_idx in range(pca_matrix.shape[1]):
        f, t, Sxx = spectrogram(
            pca_matrix[:, comp_idx],
            fs=fs,
            window=window,
            nperseg=nperseg,
            noverlap=noverlap,
            scaling='density'   # Power Spectral Density (V²/Hz)
        )
        if freqs is None:
            freqs = f
            times = t
        spectrograms.append(Sxx)
    
    return freqs, times, spectrograms

def crop_freq_band(freqs, spectrograms, fmin, fmax):
    """Cắt dải tần quan tâm từ spectrogram."""
    mask = (freqs >= fmin) & (freqs <= fmax)
    cropped_freqs = freqs[mask]
    cropped_specs = [Sxx[mask, :] for Sxx in spectrograms]
    return cropped_freqs, cropped_specs

def to_decibel(spectrograms, ref=1e-12):
    """Chuyển Power Spectrogram sang dB: 10*log10(Sxx/ref)."""
    return [10 * np.log10(np.maximum(Sxx, ref) / ref) for Sxx in spectrograms]

# ============ Tính Spectrogram cho Amplitude PCA ============
spec_freqs, spec_times, spec_amp_full = compute_spectrogram_matrix(
    amp_pca, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP, window=WINDOW
)

# ============ Tính Spectrogram cho Phase PCA ============
_, _, spec_phs_full = compute_spectrogram_matrix(
    phs_pca, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP, window=WINDOW
)

print(f"Full Spectrogram:")
print(f"  Frequency bins: {len(spec_freqs)} ({spec_freqs[0]:.3f} - {spec_freqs[-1]:.3f} Hz)")
print(f"  Freq resolution: {spec_freqs[1] - spec_freqs[0]:.4f} Hz")
print(f"  Time frames: {len(spec_times)} ({spec_times[0]:.2f} - {spec_times[-1]:.2f} s)")
print(f"  Each component shape: {spec_amp_full[0].shape} (freq_bins × time_frames)")

# ============ Cắt dải tần nhịp thở ============
breath_freqs, breath_spec_amp = crop_freq_band(spec_freqs, spec_amp_full, BREATH_FREQ_MIN, BREATH_FREQ_MAX)
_, breath_spec_phs = crop_freq_band(spec_freqs, spec_phs_full, BREATH_FREQ_MIN, BREATH_FREQ_MAX)

print(f"\nBreathing Band [{BREATH_FREQ_MIN}-{BREATH_FREQ_MAX} Hz]:")
print(f"  Frequency bins: {len(breath_freqs)}")
print(f"  Each component shape: {breath_spec_amp[0].shape}")

# ============ Chuyển sang Decibel ============
breath_spec_amp_dB = to_decibel(breath_spec_amp)
breath_spec_phs_dB = to_decibel(breath_spec_phs)

print(f"\nDecibel range (Amplitude PC1):")
print(f"  Min: {np.min(breath_spec_amp_dB[0]):.1f} dB")
print(f"  Max: {np.max(breath_spec_amp_dB[0]):.1f} dB")

# ============ Stack thành tensor CNN-ready ============
# Shape: (N_COMPONENTS, n_freq_bins, n_time_frames) — giống multi-channel image
cnn_input_amp = np.stack(breath_spec_amp_dB, axis=0)   # (5, freq, time)
cnn_input_phs = np.stack(breath_spec_phs_dB, axis=0)   # (5, freq, time)

# Hoặc concat cả amp + phase: (10, freq, time)
cnn_input_combined = np.concatenate([cnn_input_amp, cnn_input_phs], axis=0)

print(f"\n{'='*60}")
print(f"CNN Input Tensors:")
print(f"  Amplitude only:  {cnn_input_amp.shape}  (channels=PC, freq, time)")
print(f"  Phase only:      {cnn_input_phs.shape}  (channels=PC, freq, time)")
print(f"  Combined:        {cnn_input_combined.shape}  (channels=amp+phs, freq, time)")
print(f"{'='*60}")

# Lưu feature matrices
feature_matrices = {
    'breath_freqs': breath_freqs,
    'spec_times': spec_times,
    'breath_spec_amp': breath_spec_amp,
    'breath_spec_phs': breath_spec_phs,
    'breath_spec_amp_dB': breath_spec_amp_dB,
    'breath_spec_phs_dB': breath_spec_phs_dB,
    'cnn_input_amp': cnn_input_amp,
    'cnn_input_phs': cnn_input_phs,
    'cnn_input_combined': cnn_input_combined,
}

print("\nFeature matrices saved!")

Tính Power Spectrogram (STFT) cho 5 principal components...
  Window: hann, nperseg=500 (5.0s)
  Overlap: 450 (90%)
  Hop size: 50 samples (0.50s)
  Breathing band: [0.1 - 0.6] Hz

Full Spectrogram:
  Frequency bins: 251 (0.000 - 50.000 Hz)
  Freq resolution: 0.2000 Hz
  Time frames: 71 (2.50 - 37.50 s)
  Each component shape: (251, 71) (freq_bins × time_frames)

Breathing Band [0.1-0.6 Hz]:
  Frequency bins: 2
  Each component shape: (2, 71)

Decibel range (Amplitude PC1):
  Min: 88.0 dB
  Max: 135.3 dB

CNN Input Tensors:
  Amplitude only:  (5, 2, 71)  (channels=PC, freq, time)
  Phase only:      (5, 2, 71)  (channels=PC, freq, time)
  Combined:        (10, 2, 71)  (channels=amp+phs, freq, time)

Feature matrices saved!


In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

colors_pc = ['royalblue', 'darkorange', 'green', 'red', 'purple',
             'brown', 'pink', 'gray', 'olive', 'cyan']

# ============================================================
# PLOT 1: Spectrogram heatmap cho từng PC (Amplitude + Phase, dB)
# ============================================================
fig1 = make_subplots(rows=N_COMPONENTS, cols=2,
                     subplot_titles=[t for i in range(N_COMPONENTS)
                                     for t in (f'Amplitude PC{i+1} (dB)',
                                               f'Phase PC{i+1} (dB)')],
                     vertical_spacing=0.03, horizontal_spacing=0.08)

for i in range(N_COMPONENTS):
    # Amplitude Spectrogram (dB)
    fig1.add_trace(go.Heatmap(
        z=breath_spec_amp_dB[i], x=spec_times, y=breath_freqs,
        colorscale='Viridis', colorbar=dict(title='dB', len=1/N_COMPONENTS, y=1 - (i+0.5)/N_COMPONENTS),
        showscale=(i == 0), name=f'Amp PC{i+1}'
    ), row=i+1, col=1)

    # Phase Spectrogram (dB)
    fig1.add_trace(go.Heatmap(
        z=breath_spec_phs_dB[i], x=spec_times, y=breath_freqs,
        colorscale='Magma', colorbar=dict(title='dB', len=1/N_COMPONENTS, y=1 - (i+0.5)/N_COMPONENTS, x=1.07),
        showscale=(i == 0), name=f'Phs PC{i+1}'
    ), row=i+1, col=2)

    fig1.update_yaxes(title_text='Freq (Hz)', row=i+1, col=1, title_font_size=9)

fig1.update_xaxes(title_text='Time (s)', row=N_COMPONENTS, col=1)
fig1.update_xaxes(title_text='Time (s)', row=N_COMPONENTS, col=2)
fig1.update_layout(height=350*N_COMPONENTS, width=1800, showlegend=False,
                   template='plotly_white',
                   title_text='Breathing Spectrograms per PC (dB)')
fig1.show()

# ============================================================
# PLOT 2: Tổng hợp — Mean Power theo thời gian & tần số
# ============================================================
fig2 = make_subplots(rows=2, cols=2,
                     subplot_titles=[
                         'Amplitude — Mean Breathing Power over Time',
                         'Phase — Mean Breathing Power over Time',
                         'Amplitude — Mean Breathing Spectrum',
                         'Phase — Mean Breathing Spectrum',
                     ])

for i in range(N_COMPONENTS):
    # Mean power over time (amplitude)
    mean_power_t = np.mean(breath_spec_amp_dB[i], axis=0)
    fig2.add_trace(go.Scatter(x=spec_times, y=mean_power_t, mode='lines',
                               line=dict(color=colors_pc[i], width=1.2), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='amp_t',
                               showlegend=True), row=1, col=1)

    # Mean power over time (phase)
    mean_power_t = np.mean(breath_spec_phs_dB[i], axis=0)
    fig2.add_trace(go.Scatter(x=spec_times, y=mean_power_t, mode='lines',
                               line=dict(color=colors_pc[i], width=1.2), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='phs_t',
                               showlegend=False), row=1, col=2)

    # Mean power over frequency (amplitude)
    mean_power_f = np.mean(breath_spec_amp_dB[i], axis=1)
    fig2.add_trace(go.Scatter(x=breath_freqs, y=mean_power_f, mode='lines',
                               line=dict(color=colors_pc[i], width=1.5), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='amp_f',
                               showlegend=False), row=2, col=1)

    # Mean power over frequency (phase)
    mean_power_f = np.mean(breath_spec_phs_dB[i], axis=1)
    fig2.add_trace(go.Scatter(x=breath_freqs, y=mean_power_f, mode='lines',
                               line=dict(color=colors_pc[i], width=1.5), opacity=0.8,
                               name=f'PC{i+1}', legendgroup='phs_f',
                               showlegend=False), row=2, col=2)

fig2.update_xaxes(title_text='Time (s)', row=1, col=1)
fig2.update_xaxes(title_text='Time (s)', row=1, col=2)
fig2.update_xaxes(title_text='Frequency (Hz)', row=2, col=1)
fig2.update_xaxes(title_text='Frequency (Hz)', row=2, col=2)
fig2.update_yaxes(title_text='Mean Power (dB)', row=1, col=1)
fig2.update_yaxes(title_text='Mean Power (dB)', row=1, col=2)
fig2.update_yaxes(title_text='Mean Power (dB)', row=2, col=1)
fig2.update_yaxes(title_text='Mean Power (dB)', row=2, col=2)
fig2.update_layout(height=800, width=1600, template='plotly_white')
fig2.show()

# ============================================================
# PLOT 3: CNN Input Tensor preview (combined channels)
# ============================================================
fig3 = make_subplots(rows=2, cols=N_COMPONENTS,
                     subplot_titles=[f'Amp PC{i+1}' for i in range(N_COMPONENTS)]
                                    + [f'Phs PC{i+1}' for i in range(N_COMPONENTS)],
                     vertical_spacing=0.12, horizontal_spacing=0.04)

for i in range(N_COMPONENTS):
    # Amplitude channel
    fig3.add_trace(go.Heatmap(
        z=cnn_input_combined[i], x=spec_times, y=breath_freqs,
        colorscale='Viridis', showscale=False
    ), row=1, col=i+1)

    # Phase channel
    fig3.add_trace(go.Heatmap(
        z=cnn_input_combined[N_COMPONENTS + i], x=spec_times, y=breath_freqs,
        colorscale='Magma', showscale=False
    ), row=2, col=i+1)

    if i == 0:
        fig3.update_yaxes(title_text='Freq (Hz)', row=1, col=1)
        fig3.update_yaxes(title_text='Freq (Hz)', row=2, col=1)
    fig3.update_xaxes(title_text='Time (s)', row=2, col=i+1)

fig3.update_layout(
    height=500, width=350*N_COMPONENTS, showlegend=False,
    template='plotly_white',
    title_text=f'CNN Input Tensor — {cnn_input_combined.shape[0]} channels × '
               f'{cnn_input_combined.shape[1]} freq × {cnn_input_combined.shape[2]} time (dB)')
fig3.show()

# ============================================================
# Thống kê
# ============================================================
print(f"\n{'='*60}")
print("SPECTROGRAM STATISTICS (Breathing Band, dB)")
print(f"{'='*60}")
for i in range(N_COMPONENTS):
    print(f"\nPC{i+1}:")
    print(f"  Amplitude dB — min: {np.min(breath_spec_amp_dB[i]):.1f}, "
          f"max: {np.max(breath_spec_amp_dB[i]):.1f}, "
          f"mean: {np.mean(breath_spec_amp_dB[i]):.1f}")
    print(f"  Phase dB    — min: {np.min(breath_spec_phs_dB[i]):.1f}, "
          f"max: {np.max(breath_spec_phs_dB[i]):.1f}, "
          f"mean: {np.mean(breath_spec_phs_dB[i]):.1f}")

print(f"\n{'='*60}")
print(f"Spectrogram params: window={WINDOW} {NPERSEG/FS:.0f}s, "
      f"overlap={NOVERLAP/NPERSEG*100:.0f}%, hop={NPERSEG-NOVERLAP} samples")
print(f"CNN tensor shape: {cnn_input_combined.shape}")
print(f"{'='*60}")


SPECTROGRAM STATISTICS (Breathing Band, dB)

PC1:
  Amplitude dB — min: 88.0, max: 135.3, mean: 115.5
  Phase dB    — min: 103.9, max: 131.9, mean: 120.0

PC2:
  Amplitude dB — min: 99.4, max: 130.4, mean: 119.6
  Phase dB    — min: 105.4, max: 131.5, mean: 120.6

PC3:
  Amplitude dB — min: 92.3, max: 123.6, mean: 113.9
  Phase dB    — min: 102.0, max: 127.2, mean: 118.7

PC4:
  Amplitude dB — min: 92.8, max: 127.0, mean: 116.3
  Phase dB    — min: 93.9, max: 122.1, mean: 112.0

PC5:
  Amplitude dB — min: 95.9, max: 124.9, mean: 115.6
  Phase dB    — min: 88.8, max: 120.4, mean: 112.9

Spectrogram params: window=hann 5s, overlap=90%, hop=50 samples
CNN tensor shape: (10, 2, 71)


# 3. Phase Processing (Unwrapping & Sanitization)

In [ ]:

def unwrap_phase(phase_list):
    """Thực hiện unwrap pha cho từng packet."""
    return [np.unwrap(p) if len(p) > 0 else p for p in phase_list]

def sanitize_phase_robust(phase_array):
    """
    Loại bỏ lỗi pha tuyến tính (Linear Phase Offset) do SFO/CFO.
    Sử dụng Linear Regression trên các subcarrier có giá trị.
    """
    if len(phase_array) == 0:
        return phase_array
    
    # Chỉ lấy các subcarrier có pha khác 0 (tránh guard subcarriers / DC)
    valid_idx = np.where(np.abs(phase_array) > 1e-6)[0]
    if len(valid_idx) < 2:
        return phase_array
        
    x = valid_idx
    y = phase_array[valid_idx]
    
    # Linear fit: y = ax + b
    a, b = np.polyfit(x, y, 1)
    
    sanitized = phase_array.copy()
    sanitized[valid_idx] = y - (a * x + b)
    return sanitized

# Áp dụng cho dữ liệu
print("Đang thực hiện Phase Unwrapping...")
df_calc["phase_unwrapped"] = unwrap_phase(df_calc["phase"])

print("Đang thực hiện Phase Sanitization...")
df_calc["phase_sanitized"] = df_calc["phase_unwrapped"].apply(sanitize_phase_robust)

print("Hoàn tất xử lý pha.")


In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# So sánh phase trước và sau xử lý cho packet đầu tiên có dữ liệu
idx = df_calc["subcarrier_count"].gt(0).idxmax()

p_raw = df_calc.loc[idx, "phase"]
p_unwrapped = df_calc.loc[idx, "phase_unwrapped"]
p_sanitized = df_calc.loc[idx, "phase_sanitized"]

fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=("Raw Phase (Wrapped)", "Unwrapped Phase", "Sanitized Phase"))

x = np.arange(len(p_raw))

fig.add_trace(go.Scatter(x=x, y=p_raw, name="Raw"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=p_unwrapped, name="Unwrapped"), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=p_sanitized, name="Sanitized"), row=3, col=1)

fig.update_layout(height=800, title_text=f"Phase Processing Comparison - Packet {idx}")
fig.show()
